# 12_prep_docking — 도킹용 3D 리간드 준비

**한 줄 요약:** 스크리닝 상위 후보(prob≥0.7)의 SMILES를 **3차원 구조(3D)** 로 만들어 도킹 프로그램이 읽을 SDF 파일과 정보표(매니페스트)를 만든다.
**용어:** 3D 임베딩=평면 SMILES에 실제 공간 좌표를 부여 / MMFF·UFF=구조를 에너지적으로 안정화(최소화) / SDF=3D 분자 파일 형식.
**큰 흐름:** ① 준비 → ② 후보 읽기·3D 생성·저장 (한 셀)

> **📌 읽는 법**: 각 코드 셀은 **[① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기]**. ③은 그 셀에 **처음 나온** 함수·문법(기초 반복은 *(01에서 설명)*).

### 준비 — 폴더 위치 맞추기
어느 폴더에서 열어도 프로젝트 최상위에서 실행되도록 이동.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')  # data/ 폴더를 찾을 때까지 상위로 (하위 폴더에서 열어도 동작)
print('작업 폴더:', os.getcwd())

🔎 *(01에서 설명)*: `os.chdir('..')`=상위 폴더 이동, `print`=출력.

### 셀 1 — 후보를 3D로 만들어 SDF·매니페스트 저장
상위 후보를 하나씩 3D 구조로 변환(수소 추가 → 좌표 생성 → 에너지 최소화)해 SDF에 쓰고, 각 후보의 물성·유연성·성공여부를 표로 저장한다.

In [ ]:
import os
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

SRC = "data/HSD17B13_npass_ranked_full.csv"
OUTDIR = "data/docking"
SDF = os.path.join(OUTDIR, "hsd17b13_ligands.sdf")
MAN = os.path.join(OUTDIR, "docking_manifest.csv")
PROB_MIN = 0.7
os.makedirs(OUTDIR, exist_ok=True)

r = pd.read_csv(SRC)
cand = r[(r.active_prob >= PROB_MIN) & (~r.is_known)].reset_index(drop=True)
print(f"도킹 준비 대상: prob>={PROB_MIN} 신규 후보 {len(cand)}개")


def make_3d(smi):
    """SMILES → 3D 최소화 mol (실패 시 None)"""
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None, "parse_fail"
    mol = Chem.AddHs(mol)
    p = AllChem.ETKDGv3()
    p.randomSeed = 42
    if AllChem.EmbedMolecule(mol, p) != 0:
        p.useRandomCoords = True
        if AllChem.EmbedMolecule(mol, p) != 0:
            return None, "embed_fail"
    try:
        if AllChem.MMFFHasAllMoleculeParams(mol):
            AllChem.MMFFOptimizeMolecule(mol, maxIters=1000)
            ff = "MMFF"
        else:
            AllChem.UFFOptimizeMolecule(mol, maxIters=1000)
            ff = "UFF"
    except Exception:
        ff = "none"
    return mol, ff


writer = Chem.SDWriter(SDF)
rows, ok = [], 0
for _, x in cand.iterrows():
    mol, status = make_3d(x.canonical_smiles)
    base = Chem.MolFromSmiles(str(x.canonical_smiles))
    rot = rdMolDescriptors.CalcNumRotatableBonds(base) if base else np.nan
    mw = Descriptors.MolWt(base) if base else np.nan
    logp = Descriptors.MolLogP(base) if base else np.nan
    success = mol is not None
    if success:
        mol.SetProp("_Name", str(x.np_id))
        mol.SetProp("np_id", str(x.np_id))
        mol.SetProp("active_prob", f"{x.active_prob:.4f}")
        mol.SetProp("max_sim_known", f"{x.max_sim_known:.4f}")
        writer.write(mol)
        ok += 1
    rows.append({
        "np_id": x.np_id, "active_prob": round(x.active_prob, 4),
        "max_sim_known": round(x.max_sim_known, 4),
        "MW": round(mw, 1) if mw == mw else None,
        "logP": round(logp, 2) if logp == logp else None,
        "n_rotatable": rot,
        "flexible_warn": (rot is not np.nan and rot > 10),  # Vina 유연성 한계
        "embed_status": status if not success else status,  # ff명 or 실패사유
        "canonical_smiles": x.canonical_smiles,
    })
writer.close()

man = pd.DataFrame(rows)
man.to_csv(MAN, index=False)

print(f"3D 생성 성공 {ok}/{len(cand)} → {SDF}")
fail = man[~man.embed_status.isin(["MMFF", "UFF", "none"])]
if len(fail):
    print(f"3D 실패 {len(fail)}개(대개 거대·복잡 천연물): "
          + ", ".join(fail.np_id.tolist()))
flex = man[man.flexible_warn == True]
print(f"고유연성(회전결합>10, 도킹 신뢰 낮음) {len(flex)}개")
print(f"매니페스트: {MAN}")
print("\n=== 상위 10 (도킹 우선순위) ===")
print(man.head(10)[["np_id", "active_prob", "max_sim_known", "MW", "logP",
                    "n_rotatable", "embed_status"]].to_string(index=False))

🔎 **코드 뜯어보기 (셀 1)**
- `from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors` : 3D 생성·물성 계산 도구.
- `r[(r.active_prob >= 0.7) & (~r.is_known)]` : 확률 0.7 이상이고 신규인 후보만 고르기.
- `def make_3d(smi):` : SMILES를 3D로 만드는 함수. `Chem.AddHs(mol)`=수소 원자 추가(3D에 필요), `AllChem.ETKDGv3()`=3D 좌표 생성 설정, `AllChem.EmbedMolecule(mol, p)`=실제 3D 좌표 만들기(성공하면 0).
- `AllChem.MMFFOptimizeMolecule(mol)` : 힘장(MMFF)으로 구조를 에너지 최소화(자연스러운 모양으로). 안 되면 UFF 사용.
- `Chem.SDWriter(SDF)` : SDF 파일에 3D 분자를 쓰는 도구. `writer.write(mol)`=한 분자 쓰기, `writer.close()`=마무리.
- `for _, x in cand.iterrows():` : 후보 표를 한 줄씩. `mol.SetProp("np_id", ...)`=분자에 정보 꼬리표 달기.
- `rdMolDescriptors.CalcNumRotatableBonds(base)` : 회전 가능한 결합 수(많으면 도킹이 어려움 → flexible_warn). `mw == mw`=빈 값(NaN) 판별(NaN은 자기 자신과 다름).